# 📈 Experiment Result Comparison

This notebook loads all evaluation results recorded in MLflow and compares them.

## Comparison Axes

| Axis | Variants |
|------|----------|
| **Model** | Base vs LoRA vs OSFT |
| **Knowledge Access** | no_knowledge vs RAG |
| **Mode** | simple_rag vs agent_rag |

## Comparison Metrics

- **Task success rate** (τ episodes): official pass^k
- **Diagnostic accuracy**: offline holdout results (lab-derived)
- **Retention delta**: ARC-Challenge accuracy change (retention_delta_pp)
- **Latency**: mean episode response time
- **Failure analysis**: error classification by type

> ⚠️ This comparison is based on actual MLflow results.  
> No fabricated performance numbers are inserted.

In [ ]:
"""Load all evaluation runs from MLflow."""

import os
import json
from pathlib import Path

import pandas as pd

from rhoai_model_training_lab.config import load_env, load_eval_config, PROJECT_ROOT

load_env()

eval_config = load_eval_config()
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_EVAL", "rhoai-model-training-lab-evaluation")

runs_df = None

if mlflow_uri:
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        experiment = mlflow.get_experiment_by_name(experiment_name)

        if experiment:
            runs = mlflow.search_runs(
                experiment_ids=[experiment.experiment_id],
                order_by=["start_time DESC"],
            )
            runs_df = runs
            print(f"✅ Loaded {len(runs)} runs from MLflow")
            print(f"   Experiment: {experiment_name}")
            print(f"   URI: {mlflow_uri}")
        else:
            print(f"⚠️  Experiment '{experiment_name}' not found.")
    except Exception as exc:
        print(f"❌ Failed to load from MLflow: {exc}")
else:
    print("⚠️  MLFLOW_TRACKING_URI not set")

# Fallback: load from local result files
if runs_df is None or len(runs_df) == 0:
    print("\n📂 Attempting to load from local result files...")
    results_dir = PROJECT_ROOT / eval_config["general"]["output_dir"]
    local_results = []

    for rf in sorted(results_dir.glob("**/*.json")):
        try:
            with open(rf) as f:
                data = json.load(f)
            if isinstance(data, dict):
                data["_source_file"] = str(rf.relative_to(results_dir))
                local_results.append(data)
        except Exception:
            continue

    if local_results:
        runs_df = pd.DataFrame(local_results)
        print(f"  Loaded {len(local_results)} local result files")
    else:
        print("  ⚠️  No result files found.")
        print("  Please run 07_evaluate.ipynb first.")

if runs_df is not None and len(runs_df) > 0:
    print(f"\nTotal {len(runs_df)} run records available")

In [ ]:
"""Build comparison table: Base vs LoRA vs OSFT × knowledge access."""

from rich.console import Console
from rich.table import Table

console = Console()

print("=" * 70)
print("📊 Experiment Comparison Table")
print("=" * 70)

# Define expected variants
expected_variants = [
    {"variant": "base_no_knowledge", "model": "Base", "knowledge": "No KB"},
    {"variant": "base_rag", "model": "Base", "knowledge": "RAG"},
    {"variant": "lora_no_knowledge", "model": "LoRA", "knowledge": "No KB"},
    {"variant": "lora_rag", "model": "LoRA", "knowledge": "RAG"},
    {"variant": "osft_no_knowledge", "model": "OSFT", "knowledge": "No KB"},
    {"variant": "osft_rag", "model": "OSFT", "knowledge": "RAG"},
]

table = Table(title="Model × Knowledge Access Comparison", show_header=True)
table.add_column("Variant", style="bold")
table.add_column("Model")
table.add_column("Knowledge Access")
table.add_column("Task Success Rate")
table.add_column("Diagnostic Accuracy")
table.add_column("Retention Δpp")
table.add_column("Mean Latency (s)")
table.add_column("Failures")

if runs_df is not None and len(runs_df) > 0:
    for ev in expected_variants:
        variant = ev["variant"]
        # Try to find matching run
        metric_cols = [c for c in runs_df.columns if "metric" in c.lower() or variant in str(c).lower()]

        # Extract metrics (adapt to actual MLflow column naming)
        task_sr = "N/A"
        diag_acc = "N/A"
        retention = "N/A"
        latency = "N/A"
        failures = "N/A"

        # Look for matching data in runs
        for col_prefix in ["metrics.", "params."]:
            if f"{col_prefix}variant" in runs_df.columns:
                mask = runs_df[f"{col_prefix}variant"] == variant
                matched = runs_df[mask]
                if len(matched) > 0:
                    row = matched.iloc[0]
                    task_sr = f"{row.get('metrics.task_success_rate', 'N/A')}"
                    diag_acc = f"{row.get('metrics.answer_accuracy', 'N/A')}"
                    retention = f"{row.get('metrics.retention_delta_pp', 'N/A')}"
                    latency = f"{row.get('metrics.mean_wall_time_seconds', 'N/A')}"
                    failures = f"{row.get('metrics.failed', 'N/A')}"

        table.add_row(
            variant, ev["model"], ev["knowledge"],
            task_sr, diag_acc, retention, latency, failures,
        )
else:
    for ev in expected_variants:
        table.add_row(
            ev["variant"], ev["model"], ev["knowledge"],
            "—", "—", "—", "—", "—",
        )

console.print(table)

if runs_df is None or len(runs_df) == 0:
    print("\n⚠️  No evaluation data available — the table is empty.")
    print("   Please run 07_evaluate.ipynb first.")

In [ ]:
"""Plot task success rates."""

try:
    import matplotlib.pyplot as plt
    import numpy as np

    print("=" * 70)
    print("📊 Task Success Rate Visualization")
    print("=" * 70)

    models = ["Base", "LoRA", "OSFT"]
    no_kb_rates = []
    rag_rates = []

    # Extract data from runs or use placeholders
    for model in models:
        no_kb_key = f"{model.lower()}_no_knowledge"
        rag_key = f"{model.lower()}_rag"

        no_kb_val = 0.0
        rag_val = 0.0

        if runs_df is not None and len(runs_df) > 0:
            for _, row in runs_df.iterrows():
                v = row.get("params.variant", row.get("variant", ""))
                sr = row.get("metrics.task_success_rate", row.get("task_success_rate", None))
                if sr is not None and not pd.isna(sr):
                    if v == no_kb_key:
                        no_kb_val = float(sr)
                    elif v == rag_key:
                        rag_val = float(sr)

        no_kb_rates.append(no_kb_val)
        rag_rates.append(rag_val)

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, no_kb_rates, width, label="No Knowledge", color="#90CAF9")
    bars2 = ax.bar(x + width/2, rag_rates, width, label="RAG", color="#FF8A65")

    ax.set_ylabel("Task Success Rate")
    ax.set_title("Task Success Rate by Model × Knowledge Access")
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f"{height:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()

    if all(v == 0 for v in no_kb_rates + rag_rates):
        print("⚠️  All success rates are 0 — it appears there is no evaluation data yet.")

except ImportError:
    print("matplotlib is not installed, skipping visualization.")
    print("Run: pip install matplotlib")

In [ ]:
"""Plot retention delta."""

try:
    import matplotlib.pyplot as plt
    import numpy as np

    print("=" * 70)
    print("📊 Retention Delta Visualization")
    print("=" * 70)

    models = ["LoRA", "OSFT"]
    deltas = []
    base_acc = 0.0

    for model in models:
        delta = 0.0
        if runs_df is not None and len(runs_df) > 0:
            for _, row in runs_df.iterrows():
                v = row.get("params.variant", row.get("variant", ""))
                d = row.get("metrics.retention_delta_pp", row.get("retention_delta_pp", None))
                if d is not None and not pd.isna(d) and model.lower() in str(v).lower():
                    delta = float(d)
                ba = row.get("metrics.base_accuracy", row.get("base_accuracy", None))
                if ba is not None and not pd.isna(ba) and "base" in str(v).lower():
                    base_acc = float(ba)
        deltas.append(delta)

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
    bars = ax.bar(models, deltas, color=colors, width=0.5)

    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="-")
    ax.set_ylabel("Retention Delta (pp)")
    ax.set_title(f"ARC-Challenge Retention Delta\n(Base model accuracy: {base_acc:.1%})")
    ax.grid(axis="y", alpha=0.3)

    for bar, delta in zip(bars, deltas):
        y_pos = bar.get_height() + (0.3 if delta >= 0 else -0.5)
        ax.text(bar.get_x() + bar.get_width()/2., y_pos,
                f"{delta:+.1f}pp", ha="center", va="bottom" if delta >= 0 else "top")

    plt.tight_layout()
    plt.show()

    print("retention_delta_pp = 100 × (adapted model accuracy - base model accuracy)")
    print("Positive = existing capability improved, Negative = existing capability degraded (forgetting)")

except ImportError:
    print("matplotlib is not installed, skipping visualization.")

In [ ]:
"""Analyze failures and latency."""

print("=" * 70)
print("🔍 Failure Analysis and Latency")
print("=" * 70)

if runs_df is not None and len(runs_df) > 0:
    # Failure analysis
    print("\n--- Failure Type Analysis ---")
    error_cols = [c for c in runs_df.columns if "error" in c.lower() or "fail" in c.lower()]
    if error_cols:
        for col in error_cols:
            values = runs_df[col].dropna()
            if len(values) > 0:
                print(f"\n  {col}:")
                if values.dtype in ["int64", "float64"]:
                    print(f"    Total: {values.sum():.0f}")
                    print(f"    Mean: {values.mean():.1f}")
                else:
                    for v, count in values.value_counts().items():
                        print(f"    {v}: {count}")
    else:
        print("  No failure-related columns found.")

    # Latency analysis
    print("\n--- Latency Analysis ---")
    latency_cols = [c for c in runs_df.columns if "latency" in c.lower() or "time" in c.lower()]
    if latency_cols:
        for col in latency_cols:
            values = runs_df[col].dropna()
            if len(values) > 0 and values.dtype in ["int64", "float64"]:
                print(f"\n  {col}:")
                print(f"    Mean: {values.mean():.2f}")
                print(f"    Median: {values.median():.2f}")
                print(f"    Min: {values.min():.2f}")
                print(f"    Max: {values.max():.2f}")
    else:
        print("  No latency-related columns found.")

    # Token usage
    print("\n--- Token Usage ---")
    token_cols = [c for c in runs_df.columns if "token" in c.lower()]
    if token_cols:
        for col in token_cols:
            values = runs_df[col].dropna()
            if len(values) > 0 and values.dtype in ["int64", "float64"]:
                print(f"  {col}: mean={values.mean():.0f}, total={values.sum():.0f}")
    else:
        print("  No token usage columns found.")
else:
    print("⚠️  No data available for analysis.")

In [ ]:
"""Summary and conclusions."""

from rich.panel import Panel

print("=" * 70)
print("📝 Summary and Conclusions")
print("=" * 70)

summary_text = """
🏦 τ-Knowledge Banking Model Training Experiment Results Summary

Experiment Setup:
  • Base model: Qwen/Qwen3-4B-Instruct-2507
  • Domain: τ-Knowledge banking_knowledge
  • Training methods: LoRA (r=16, α=32) / OSFT (unfreeze=0.25)
  • Evaluation: Diagnostics (offline) + τ episodes (online) + Retention (ARC-Challenge)
"""

console.print(Panel(summary_text, title="Experiment Summary", border_style="blue"))

# Conclusions based on available data
print("\nKey Observations:")
print()

if runs_df is not None and len(runs_df) > 0:
    print("  1. Model Performance Comparison:")
    print("     - Refer to the tables and charts above for specific numbers.")
    print("     - Benchmark performance conclusions cannot be drawn from small-scale smoke tests.")
    print()
    print("  2. KB Adaptation Experiment (kb_adaptation):")
    print("     - This experiment measures the ability to learn KB and apply it to new situations.")
    print("     - It does not claim equivalence to training-free leaderboard protocols.")
    print()
    print("  3. Retention Analysis:")
    print("     - A small sample from a single benchmark cannot prove forgetting elimination.")
    print("     - Causal claims that OSFT has better retention than LoRA")
    print("       cannot hold without a full SFT control group.")
else:
    print("  ⚠️  No evaluation data available.")
    print("     Please run 07_evaluate.ipynb to complete the evaluation.")

print("\nLimitations and Caveats:")
print("  • Synthetic single-turn diagnostic accuracy ≠ official τ multi-turn task success")
print("  • pass^k (consistent success) ≠ pass@k (best attempt)")
print("  • Small models (4B) have inherent limitations in full agent task performance")
print("  • Data preparation quality significantly affects training effectiveness")

print("\nNext Steps:")
print("  • Run more episodes and trials for statistical significance")
print("  • Improve training data quality (see data_preparation/ notebooks)")
print("  • Hyperparameter tuning (based on dev data)")
print("  • Full SFT control group experiment (optional)")